
# Ingesta SharePoint → Bronze

## Propósito

Extraer los datos desde SharePoint correspondientes al corte previamente validado por `00_prepare_pipeline_control` y almacenarlos en la capa Bronze en formato Delta, conservando la trazabilidad de la ejecución y del archivo de origen.

## Flujo

1. Obtener `fecha_inicio`, `fecha_fin` e `id_ejecucion` mediante Task Values.
2. Validar los parámetros recibidos.
3. Autenticar el acceso a SharePoint.
4. Descargar la fuente de datos al área RAW.
5. Leer y procesar los datos con PySpark.
6. Aplicar el corte de extracción.
7. Normalizar los nombres de las columnas.
8. Agregar los metadatos de trazabilidad.
9. Validar que el corte no haya sido cargado previamente.
10. Escribir los datos en Bronze en formato Delta.

## Origen

SharePoint.

**Archivo de origen:**

`DataCoSupplyChainDataset.csv`

## Destino

Capa Bronze.

**Tabla:**

`brz_dataco.sharepoint.raw_dataco_supply_chain`

## Parámetros recibidos

* `fecha_inicio`
* `fecha_fin`
* `id_ejecucion`

Estos valores son publicados por `00_prepare_pipeline_control` mediante Task Values.

## Trazabilidad generada

La carga Bronze conserva los siguientes metadatos:

* `fecha_inicio_corte`
* `fecha_fin_corte`
* `fecha_carga`
* `id_ejecucion`
* `origen`
* `archivo_origen`

## Dependencias

Este notebook debe ejecutarse como una **Task dentro de un Databricks Workflow**, después de `00_prepare_pipeline_control`, para poder consumir los valores publicados mediante Task Values.

## Resultado esperado

Una carga Bronze correspondiente únicamente al rango de fechas recibido desde `00_prepare_pipeline_control`, almacenada en formato Delta y con la trazabilidad completa de la ejecución y del archivo de origen.


In [0]:
# ============================================================
# OBTENCIÓN DE PARÁMETROS DE EXTRACCIÓN
# ============================================================

fecha_inicio = dbutils.jobs.taskValues.get(
    taskKey="00_prepare_pipeline_control",
    key="fecha_inicio",
    debugValue=""
)

fecha_fin = dbutils.jobs.taskValues.get(
    taskKey="00_prepare_pipeline_control",
    key="fecha_fin",
    debugValue=""
)

id_ejecucion = dbutils.jobs.taskValues.get(
    taskKey="00_prepare_pipeline_control",
    key="id_ejecucion",
    debugValue=""
)

print(f"Fecha inicio: {fecha_inicio}")
print(f"Fecha fin:    {fecha_fin}")
print(f"ID ejecución: {id_ejecucion}")

In [0]:
from datetime import datetime

fecha_inicio_dt = datetime.strptime(fecha_inicio, "%Y-%m-%d").date()
fecha_fin_dt = datetime.strptime(fecha_fin, "%Y-%m-%d").date()

if fecha_inicio_dt > fecha_fin_dt:
    raise ValueError(
        f"Rango de fechas inválido: "
        f"fecha_inicio={fecha_inicio}, fecha_fin={fecha_fin}"
    )

print("Parámetros de extracción válidos.")
print(f"Rango: {fecha_inicio} → {fecha_fin}")

In [0]:
# ============================================================
# VALIDACIÓN DEL ID DE EJECUCIÓN
# ============================================================

if not id_ejecucion:
    raise ValueError(
        "No se recibió el id_ejecucion desde "
        "00_prepare_pipeline_control."
    )

print("ID de ejecución recibido correctamente.")
print(f"ID ejecución: {id_ejecucion}")

In [0]:
# ============================================================
# OBTENCIÓN DE PARÁMETROS DE SHAREPOINT
# ============================================================

dbutils.widgets.text("SP_TENANT_ID", "")
dbutils.widgets.text("SP_CLIENT_ID", "")
dbutils.widgets.text("SP_CLIENT_SECRET", "")

tenant_id = dbutils.widgets.get("SP_TENANT_ID")
client_id = dbutils.widgets.get("SP_CLIENT_ID")
client_secret = dbutils.widgets.get("SP_CLIENT_SECRET")


print(f"Tenant ID recibido: {tenant_id}")
print(f"Client ID recibido: {client_id}")
print("Client Secret recibido:", "Sí" if client_secret else "No")

In [0]:
# ============================================================
# AUTENTICACIÓN MICROSOFT GRAPH
# ============================================================

import requests

token_url = f"https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token"

token_data = {
    "grant_type": "client_credentials",
    "client_id": client_id,
    "client_secret": client_secret,
    "scope": "https://graph.microsoft.com/.default"
}

response = requests.post(token_url, data=token_data)

if response.status_code != 200:
    raise RuntimeError(
        f"Error al obtener token de Microsoft Graph: "
        f"{response.status_code} - {response.text}"
    )

access_token = response.json()["access_token"]

print("Token de Microsoft Graph obtenido correctamente.")

In [0]:
# ============================================================
# OBTENER SITE ID DE SHAREPOINT
# ============================================================

headers = {
    "Authorization": f"Bearer {access_token}"
}

site_url = (
    "https://graph.microsoft.com/v1.0/"
    "sites/technologicalsolutionsbpo.sharepoint.com:"
    "/sites/Storage-OnPremise2"
)

response = requests.get(site_url, headers=headers)

if response.status_code != 200:
    raise RuntimeError(
        f"Error al obtener SharePoint Site ID: "
        f"{response.status_code} - {response.text}"
    )

site_data = response.json()

site_id = site_data["id"]
site_name = site_data.get("displayName", "Sin nombre")

print(f"SharePoint encontrado: {site_name}")
print("Site ID obtenido correctamente.")

In [0]:
# ============================================================
# VALIDAR ACCESO AL ARCHIVO DE SHAREPOINT
# ============================================================

NOMBRE_ARCHIVO = "DataCoSupplyChainDataset.csv"
RUTA_CARPETA = "3. Reporte Performance Logistico DataCo"

ruta_archivo = f"{RUTA_CARPETA}/{NOMBRE_ARCHIVO}"

file_url = (
    f"https://graph.microsoft.com/v1.0/"
    f"sites/{site_id}/drive/root:/{ruta_archivo}"
)

response = requests.get(file_url, headers=headers)

if response.status_code != 200:
    raise RuntimeError(
        f"Error al localizar el archivo: "
        f"{response.status_code} - {response.text}"
    )

file_data = response.json()

print(f"Archivo encontrado: {file_data.get('name')}")
print("Acceso al archivo validado correctamente.")

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS brz_dataco;

CREATE SCHEMA IF NOT EXISTS brz_dataco.sharepoint;

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.raw.sharepoint;

In [0]:
# ============================================================
# DESCARGAR CSV COMPLETO A RAW
# ============================================================

RUTA_RAW = "/Volumes/workspace/raw/sharepoint"
RUTA_RAW_ARCHIVO = f"{RUTA_RAW}/{NOMBRE_ARCHIVO}"

file_content_url = (
    f"https://graph.microsoft.com/v1.0/"
    f"sites/{site_id}/drive/root:/{ruta_archivo}:/content"
)

response = requests.get(
    file_content_url,
    headers=headers,
    stream=True
)

if response.status_code != 200:
    raise RuntimeError(
        f"Error al descargar el archivo: "
        f"{response.status_code} - {response.text}"
    )

with open(RUTA_RAW_ARCHIVO, "wb") as f:
    for chunk in response.iter_content(chunk_size=1024 * 1024):
        if chunk:
            f.write(chunk)

print(f"Archivo descargado correctamente:")
print(RUTA_RAW_ARCHIVO)

In [0]:
display(dbutils.fs.ls("/Volumes/workspace/raw/SharePoint"))

In [0]:
# ============================================================
# CONFIGURACIÓN DE RUTA RAW
# ============================================================

NOMBRE_ARCHIVO = "DataCoSupplyChainDataset.csv"

RUTA_RAW = "/Volumes/workspace/raw/SharePoint"
RUTA_RAW_ARCHIVO = f"{RUTA_RAW}/{NOMBRE_ARCHIVO}"

print(f"Ruta RAW: {RUTA_RAW_ARCHIVO}")


# ============================================================
# LECTURA DEL CSV DESDE RAW
# ============================================================

df_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("encoding", "ISO-8859-1")
    .option("quote", '"')
    .option("escape", '"')
    .csv(RUTA_RAW_ARCHIVO)
)

print(f"Cantidad de columnas: {len(df_raw.columns)}")
print("Columnas:")
print(df_raw.columns)

display(df_raw.limit(5))

In [0]:
df_raw.printSchema()

In [0]:
# ============================================================
# CONVERSIÓN Y FILTRO DE FECHA DE EXTRACCIÓN
# ============================================================

from pyspark.sql import functions as F

COLUMNA_FECHA = "order date (DateOrders)"

df_bronze = (
    df_raw
    .withColumn(
        "order_date",
        F.to_timestamp(
            F.col(COLUMNA_FECHA),
            "M/d/yyyy H:mm"
        ).cast("date")
    )
    .filter(
        (F.col("order_date") >= F.to_date(F.lit(fecha_inicio))) &
        (F.col("order_date") <= F.to_date(F.lit(fecha_fin)))
    )
)

print(f"Registros RAW:   {df_raw.count()}")
print(f"Registros corte: {df_bronze.count()}")

In [0]:
# ============================================================
# AGREGAR METADATOS DEL CORTE Y TRAZABILIDAD
# ============================================================

df_bronze = (
    df_bronze
    .withColumn(
        "fecha_inicio_corte",
        F.to_date(F.lit(fecha_inicio))
    )
    .withColumn(
        "fecha_fin_corte",
        F.to_date(F.lit(fecha_fin))
    )
    .withColumn(
        "fecha_carga",
        F.current_timestamp()
    )
    .withColumn(
        "id_ejecucion",
        F.lit(id_ejecucion)
    )
    .withColumn(
        "origen",
        F.lit("sharepoint")
    )
    .withColumn(
        "archivo_origen",
        F.lit(NOMBRE_ARCHIVO)
    )
)

In [0]:
# ============================================================
# VALIDACIÓN DEL ESQUEMA DE TRAZABILIDAD
# ============================================================

df_bronze.select(
    "fecha_inicio_corte",
    "fecha_fin_corte",
    "fecha_carga",
    "id_ejecucion",
    "origen",
    "archivo_origen"
).printSchema()

In [0]:
# ============================================================
# NORMALIZACIÓN DE NOMBRES DE COLUMNAS PARA BRONZE
# ============================================================

import re

def normalizar_nombre_columna(nombre):
    nombre = nombre.strip().lower()
    nombre = re.sub(r"[^a-z0-9]+", "_", nombre)
    nombre = re.sub(r"_+", "_", nombre)
    nombre = nombre.strip("_")
    return nombre


df_bronze = df_bronze.toDF(
    *[normalizar_nombre_columna(c) for c in df_bronze.columns]
)

print("Nombres de columnas normalizados correctamente.")
print(df_bronze.columns)

In [0]:
display(
    df_bronze
    .select(
        "order_date_dateorders",
        "order_date",
        "order_id"
    )
    .orderBy("order_date")
    .limit(20)
)

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.bronze;

In [0]:
# ============================================================
# CONTROL DE EXISTENCIA DEL CORTE EN BRONZE
# ============================================================

TABLA_BRONZE = "brz_dataco.sharepoint.raw_dataco_supply_chain"
tabla_existe = spark.catalog.tableExists(TABLA_BRONZE)

if tabla_existe:

    df_corte_existente = (
        spark.table(TABLA_BRONZE)
        .filter(
            (F.col("fecha_inicio_corte") == F.to_date(F.lit(fecha_inicio))) &
            (F.col("fecha_fin_corte") == F.to_date(F.lit(fecha_fin)))
        )
        .limit(1)
    )

    corte_existe = df_corte_existente.count() > 0

else:
    corte_existe = False

# ============================================================
# VALIDACIÓN DEL ESQUEMA ACTUAL DE BRONZE
# ============================================================

spark.table(
    "brz_dataco.sharepoint.raw_dataco_supply_chain"
).printSchema()

# ============================================================
# ESCRITURA DEL CORTE EN BRONZE
# ============================================================

if corte_existe:

    print(
        f"El corte {fecha_inicio} → {fecha_fin} "
        f"ya existe en Bronze. No se insertarán registros."
    )

else:

    (
        df_bronze
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(TABLA_BRONZE)
    )

    print(
        f"Corte {fecha_inicio} → {fecha_fin} "
        f"insertado correctamente en Bronze."
    )

    print(f"Registros insertados: {df_bronze.count()}")

In [0]:
display(
    spark.table(TABLA_BRONZE)
    .select(
        "fecha_inicio_corte",
        "fecha_fin_corte",
        "order_date",
        "order_id"
    )
    .orderBy("order_date")
    .limit(20)
)

In [0]:
spark.table(
    "brz_dataco.sharepoint.raw_dataco_supply_chain"
).printSchema()